# Clase 134 — YOLOv11 detección + segmentación

`ultralytics` descarga modelos ~6 MB (yolo11n) — funciona en CPU pero lento. Fallback: sliding-window + HOG features + SVM sklearn.

In [ ]:
USE_YOLO = False
try:
    from ultralytics import YOLO
    USE_YOLO = True
    print('ultralytics disponible')
except Exception as e:
    print('YOLO no disponible. Fallback sliding-window HOG+SVM. Motivo:', type(e).__name__)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)

## 1. Imagen sintética con 'objetos' (círculos)

In [ ]:
img = np.full((256, 256), 0.2, dtype=np.float32)
xx, yy = np.meshgrid(np.arange(256), np.arange(256))
objects_gt = [(60, 60, 25), (180, 80, 30), (130, 190, 20)]   # (cx, cy, r)
for cx, cy, r in objects_gt:
    img[(xx-cx)**2 + (yy-cy)**2 < r**2] = 0.9
img += np.random.normal(0, 0.05, img.shape); img = np.clip(img, 0, 1)
plt.imshow(img, cmap='gray'); plt.title('3 círculos GT'); plt.axis('off'); plt.show()

## 2. Fallback: sliding-window + HOG + SVM

In [ ]:
from skimage.feature import hog
from sklearn.svm import LinearSVC

# Generar training: parches con círculo (pos) y sin (neg)
def make_patch(has_obj, sz=48):
    p = np.full((sz, sz), 0.2)
    p += np.random.normal(0, 0.05, p.shape)
    if has_obj:
        ux, vy = np.meshgrid(np.arange(sz), np.arange(sz))
        r = sz//3
        p[(ux - sz//2)**2 + (vy - sz//2)**2 < r**2] = 0.9
    return np.clip(p, 0, 1)

X_tr = np.array([hog(make_patch(i % 2 == 0), pixels_per_cell=(8,8)) for i in range(200)])
y_tr = np.array([1 if i % 2 == 0 else 0 for i in range(200)])
svm = LinearSVC(random_state=42, max_iter=5000).fit(X_tr, y_tr)
print(f'SVM train acc: {svm.score(X_tr, y_tr):.3f}')

## 3. Sliding window sobre la imagen

In [ ]:
detections = []
sz, stride = 48, 12
for y in range(0, 256-sz, stride):
    for x in range(0, 256-sz, stride):
        patch = img[y:y+sz, x:x+sz]
        feat = hog(patch, pixels_per_cell=(8,8)).reshape(1, -1)
        score = svm.decision_function(feat)[0]
        if score > 0.3:
            detections.append((x, y, sz, sz, score))
print(f'detecciones crudas: {len(detections)}')

# NMS simple por overlap > 0.3
def iou(b1, b2):
    x1,y1,w1,h1 = b1[:4]; x2,y2,w2,h2 = b2[:4]
    xa, ya = max(x1, x2), max(y1, y2)
    xb, yb = min(x1+w1, x2+w2), min(y1+h1, y2+h2)
    inter = max(0, xb-xa) * max(0, yb-ya)
    union = w1*h1 + w2*h2 - inter
    return inter / max(union, 1e-9)

detections = sorted(detections, key=lambda d: -d[4])
keep = []
for d in detections:
    if all(iou(d, k) < 0.3 for k in keep): keep.append(d)
print(f'después NMS: {len(keep)}')

## 4. Visualizar boxes detectados vs GT

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img, cmap='gray')
for cx, cy, r in objects_gt:
    ax.add_patch(patches.Rectangle((cx-r, cy-r), 2*r, 2*r, fill=False, edgecolor='lime', linewidth=2, label='GT'))
for x, y, w, h, s in keep:
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=1.5))
ax.set_title(f'GT (verde) vs detecciones (rojo) — {len(keep)} predichos'); ax.axis('off')
plt.show()

## 5. API conceptual YOLOv11

```python
from ultralytics import YOLO
model = YOLO('yolo11n.pt')             # nano (~6 MB) — también: s, m, l, x
results = model('image.jpg', conf=0.25, iou=0.45)
for r in results:
    boxes = r.boxes.xyxy        # (N, 4)  pixel coords
    classes = r.boxes.cls       # (N,)    class ids
    scores = r.boxes.conf       # (N,)    confidence
    # segmentación:
    masks = r.masks.data if r.masks else None   # con yolo11n-seg.pt

# Train tuyo:
model.train(data='coco128.yaml', epochs=50, imgsz=640)
```

Variantes: `yolo11n.pt` (det), `yolo11n-seg.pt` (segmentación), `yolo11n-pose.pt` (keypoints), `-obb` (rotated boxes).

## 6. mAP@50 manual sobre nuestras detecciones

In [ ]:
# mAP@50 = AP a IoU threshold 0.5; AP = área bajo curva precision-recall
gt_boxes = [(cx-r, cy-r, 2*r, 2*r) for cx, cy, r in objects_gt]
det_sorted = sorted(keep, key=lambda d: -d[4])
matched = [False]*len(gt_boxes); tp = []; fp = []
for d in det_sorted:
    best_iou, best_j = 0, -1
    for j, g in enumerate(gt_boxes):
        if matched[j]: continue
        i = iou(d, g)
        if i > best_iou: best_iou, best_j = i, j
    if best_iou >= 0.5 and best_j >= 0:
        matched[best_j] = True; tp.append(1); fp.append(0)
    else:
        tp.append(0); fp.append(1)
tp = np.cumsum(tp); fp = np.cumsum(fp)
precision = tp / (tp + fp + 1e-9)
recall = tp / len(gt_boxes)
ap = np.trapz(precision, recall)
print(f'AP@0.5 ≈ {ap:.3f}  (precisión final={precision[-1]:.2f}, recall final={recall[-1]:.2f})')

## Conclusiones

- YOLOv11 (Ultralytics, 2024) = SOTA single-stage detector: detección + seg + pose + OBB.
- Pipeline: backbone CNN → neck (PAN) → head (anchor-free).
- mAP@50 más permisivo; mAP@50:95 (COCO) promedia IoU thresholds 0.5→0.95.
- En producción: ONNX/TensorRT export para 5-10x speedup.
- Para custom dataset: format YOLO (txt por imagen: `class cx cy w h` normalizado).

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — Inference de detección

`YOLO('yolo11n.pt')` sobre una imagen; `results[0].show()` dibuja cajas + labels.

In [ ]:
# from ultralytics import YOLO
# modelo = YOLO("yolo11n.pt")
# resultados = modelo("zidane.jpg")
# resultados[0].show()                       # boxes + labels
# print(resultados[0].boxes.xyxy)            # coordenadas en píxeles
print("detección: YOLO('yolo11n.pt')(img) -> results[0].boxes / .show()")

### Ejercicio 2 — Segmentación

El peso `-seg` añade máscaras por instancia en `results[0].masks`.

In [ ]:
# from ultralytics import YOLO
# modelo = YOLO("yolo11n-seg.pt")
# r = modelo("bus.jpg")[0]
# print(r.masks.data.shape)                  # (N_objetos, H, W) máscaras binarias
# r.show()
print("segmentación: yolo11n-seg.pt -> results[0].masks.data")

### Ejercicio 3 — Pose (keypoints)

El peso `-pose` estima 17 keypoints COCO por persona.

In [ ]:
# from ultralytics import YOLO
# modelo = YOLO("yolo11n-pose.pt")
# r = modelo("people.jpg")[0]
# print(r.keypoints.xy.shape)                # (N_personas, 17, 2)
# r.show()
print("pose: yolo11n-pose.pt -> results[0].keypoints.xy  (17 keypoints)")

### Ejercicio 4 — Fine-tune en dataset propio

Con 50-100 imágenes anotadas y un `ds.yaml`, el entrenamiento es una llamada.

In [ ]:
# from ultralytics import YOLO
# modelo = YOLO("yolo11n.pt")
# modelo.train(data="ds.yaml", epochs=50, imgsz=640)   # reporta mAP por época
# ds.yaml: train/val (rutas) + names: [clase0, clase1, ...]
print("fine-tune: model.train(data='ds.yaml', epochs=50, imgsz=640)")

### Ejercicio 5 — Tracking multi-objeto en video

`model.track` asigna IDs persistentes por objeto con BoT-SORT o ByteTrack.

In [ ]:
# from ultralytics import YOLO
# modelo = YOLO("yolo11n.pt")
# for r in modelo.track("video.mp4", tracker="botsort.yaml", stream=True):
#     print(r.boxes.id)                      # IDs persistentes por objeto
print("tracking: model.track('video.mp4', tracker='botsort.yaml') -> boxes.id")